In [16]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np
import pandas as pd
import xarray as xr

In [17]:
# INPUT FILES
csir_file = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/CSIR_ML6/CSIR-ML6_filtered.nc"

mercator_file = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/merged_cleaned_data.nc"

# OUTPUT FILE
output_file = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/CSIR_ML6_Merged.nc"




In [18]:
# ============================================================
# OPEN DATASETS
# ============================================================

ds_csir = xr.open_dataset(csir_file)
ds_merc = xr.open_dataset(mercator_file)

print("CSIR Dataset:")
print(ds_csir)

print("\nMercator Dataset:")
print(ds_merc)

CSIR Dataset:
<xarray.Dataset> Size: 6MB
Dimensions:    (time: 133, latitude: 36, longitude: 80)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2009-12-01 2010-01-01 ... 2020-12-01
  * latitude   (latitude) float64 288B -70.5 -69.5 -68.5 ... -37.5 -36.5 -35.5
  * longitude  (longitude) float64 640B -39.5 -38.5 -37.5 ... 37.5 38.5 39.5
Data variables:
    fgco2      (time, latitude, longitude) float64 3MB ...
    pCO2       (time, latitude, longitude) float64 3MB ...

Mercator Dataset:
<xarray.Dataset> Size: 802MB
Dimensions:    (time: 147, depth: 9, latitude: 421, longitude: 120)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2009-12-01 2010-01-01 ... 2022-02-01
  * depth      (depth) float32 36B 0.494 1.541 2.646 3.819 ... 7.93 9.573 11.4
  * latitude   (latitude) float32 2kB -70.0 -69.92 -69.83 ... -35.08 -35.0
  * longitude  (longitude) float32 480B 15.0 15.08 15.17 ... 24.75 24.83 24.92
Data variables:
    so         (time, depth, latitude, longitude) float32 267MB ..

In [19]:
# ============================================================
# CONVERT TO DATAFRAMES
# ============================================================

df_csir = ds_csir.to_dataframe().reset_index()
df_merc = ds_merc.to_dataframe().reset_index()

In [20]:
# ============================================================
# CLEAN DATA
# ============================================================

# Remove rows with missing coordinates/time
df_csir = df_csir.dropna(subset=["time", "latitude", "longitude"])
df_merc = df_merc.dropna(subset=["time", "latitude", "longitude"])

# Convert time columns to datetime
df_csir["time"] = pd.to_datetime(df_csir["time"])
df_merc["time"] = pd.to_datetime(df_merc["time"])

In [21]:
# ============================================================
# SORT DATA (required for merge_asof)
# ============================================================

df_csir = df_csir.sort_values("time")
df_merc = df_merc.sort_values("time")



In [22]:
# ============================================================
# DEFINE TOLERANCES
# ============================================================

# Time tolerance
time_tolerance = pd.Timedelta("1D")

# Spatial tolerances
lat_tolerance = 0.5
lon_tolerance = 0.5

In [23]:
# ============================================================
# CREATE MATCHING COORDINATES WITH SAME DATATYPE
# ============================================================

df_csir["lat_round"] = (
    df_csir["latitude"]
    .astype("float64")
    .round(1)
)

df_csir["lon_round"] = (
    df_csir["longitude"]
    .astype("float64")
    .round(1)
)

df_merc["lat_round"] = (
    df_merc["latitude"]
    .astype("float64")
    .round(1)
)

df_merc["lon_round"] = (
    df_merc["longitude"]
    .astype("float64")
    .round(1)
)

In [24]:
print(df_csir[["lat_round", "lon_round"]].dtypes)
print(df_merc[["lat_round", "lon_round"]].dtypes)

lat_round    float64
lon_round    float64
dtype: object
lat_round    float64
lon_round    float64
dtype: object


In [25]:
# ============================================================
# MERGE USING NEAREST TIME + CLOSE LAT/LON
# ============================================================

merged = pd.merge_asof(
    df_csir,
    df_merc,
    on="time",
    by=["lat_round", "lon_round"],
    tolerance=time_tolerance,
    direction="nearest",
    suffixes=("", "_merc")
)


In [26]:
# ============================================================
# COUNT MERGED ROWS
# ============================================================

# Rows successfully matched from Mercator dataset
matched_rows = merged["latitude_merc"].notna().sum()

# Total rows in original CSIR dataset
total_csir_rows = len(df_csir)

# Total rows in Mercator dataset
total_merc_rows = len(df_merc)

print("\n================ MERGE STATISTICS ================")
print(f"Total CSIR rows:        {total_csir_rows:,}")
print(f"Total Mercator rows:   {total_merc_rows:,}")
print(f"Successfully merged:   {matched_rows:,}")

# Percentage matched
percentage = (matched_rows / total_csir_rows) * 100

print(f"Match percentage:      {percentage:.2f}%")
print("==================================================")




================ MERGE STATISTICS ================
Total CSIR rows:        383,040
Total Mercator rows:   66,837,960
Successfully merged:   46,550
Match percentage:      12.15%


In [27]:
# ============================================================
# OPTIONAL: FILTER STRICT SPATIAL DISTANCE
# ============================================================

merged = merged[
    (np.abs(merged["latitude"] - merged["latitude_merc"]) <= lat_tolerance) &
    (np.abs(merged["longitude"] - merged["longitude_merc"]) <= lon_tolerance)
]



In [28]:
# ============================================================
# REMOVE TEMP COLUMNS
# ============================================================

merged = merged.drop(columns=["lat_round", "lon_round"])



In [29]:
# ============================================================
# CONVERT BACK TO XARRAY DATASET
# ============================================================

merged_ds = xr.Dataset.from_dataframe(merged.set_index(
    ["time", "latitude", "longitude"]
))



In [30]:
# ============================================================
# SAVE OUTPUT
# ============================================================

merged_ds.to_netcdf(output_file)

print("\nMerged NetCDF saved to:")
print(output_file)


Merged NetCDF saved to:
/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/CSIR_ML6_Merged.nc
